# Imports & Functions

In [1]:
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from importlib import reload
import data_loading  # Import the module instead of specific functions


reload(data_loading)  # Reload the module after making changes

# Now you can reference the functions directly from the reloaded module
read_csv_to_dataframe = data_loading.read_csv_to_dataframe
read_txt_to_dataframe = data_loading.read_txt_to_dataframe

/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load in dataset E

In [2]:
## making file path imports more robust

# get directory of current file
current_script_directory = Path.cwd()

# Construct path to data files given relative location
usa_string = current_script_directory  / "../data/raw/usa/"
usa_jan2002_dec2009 = usa_string / "accident_hazardous_liquid_jan2002_dec2009/accident_hazardous_liquid_jan2002_dec2009.txt"

# read data into dataframe
usa_jan2002_dec2009_raw = read_txt_to_dataframe(usa_jan2002_dec2009)

.txt file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_liquid_jan2002_dec2009/accident_hazardous_liquid_jan2002_dec2009.txt' successfully read into a DataFrame.


# Cleaning

In [3]:
# copy raw df into new df
clean_df = usa_jan2002_dec2009_raw.copy()

### substance carried

In [4]:
## focus on class_txt = "Crude Oil"
## class_txt is the main classifier, but comm provides the detailed info
clean_df = clean_df.loc[clean_df['class_txt']=="CRUDE OIL"]

# we really care about the details in the comm so once we've filtered for crude oil let's drop comm=na as they don't provide us with additional value (and cause problems later on)
clean_df = clean_df.loc[clean_df['comm'].isna()==False]

## let's map strings to binary columns; specifically, heavy, medium, light, sour, and sweet
## other mappings: HLS - heavy louisana sweet (included in if statement)
new_cols = ['heavy', 'medium', 'light', 'sour', 'sweet']
for col in new_cols:
    clean_df[col] = 0
    if ((col == 'heavy') | (col == 'sweet')):
        clean_df.loc[(clean_df['comm'].str.contains(col.upper())) | (clean_df['comm'].str.contains('HLS')), col]=1
    else:
        clean_df.loc[(clean_df['comm'].str.contains(col.upper())), col]=1

## THIS was used to finalize potential classifications
# test_df = clean_df.groupby(['comm', 'class_txt'])[['heavy', 'medium', 'light', 'sour', 'sweet']].sum().reset_index()
# test_df.to_csv('classifications.csv', index=False)

In [5]:
# total length of df
len(clean_df)

1355

In [6]:
## DESCRIPTIVE ANALYSIS
# check for incident counts by characteristic
clean_df[['heavy', 'medium', 'light', 'sour', 'sweet']].sum()

## results show that sweet make up larger proportion than other characteristics, although it is a tiny portion of the total (1355 - seen above)

heavy     10
medium     2
light      7
sour      11
sweet     25
dtype: int64

### cause of incident

In [7]:
## FROM DATA DICTIONARY:
## IF spill is less than 5 barrels, use gen_cause/gen_cause_txt (options 1-8)
## elseif spill >5 barrels, use cause/cause_txt (options 1-25)
## for reference - 42 gallons in a barrel

## IN GENERAL, the gen_cause and cause are not correctly filled in - people often filled in cause when they're not supposed to
## ATTEMPT 1: tried using more detailed value - cause - but it gave too small a sample size
## ATTEMPT 2: use more general value - gen cause - to get slightly more accurate results

test_df = clean_df.copy()
test_df = test_df[['heavy', 'medium', 'light', 'sour', 'sweet', 'gen_cause', 'cause']]
test_df['final_cause'] = test_df['gen_cause'].astype(int) ## no na's
#test_df['final_cause'] = test_df['cause'].fillna(test_df['gen_cause'])

# drop old cause cols
test_df.drop(columns=['cause', 'gen_cause'], inplace=True)


In [8]:
test_df.groupby(['final_cause']).sum()

,heavy,medium,light,sour,sweet
final_cause,,,,,
1,1,2,1,3,11
2,1,0,0,1,1
3,0,0,0,0,1
4,0,0,0,0,0
5,5,0,0,0,4
6,2,0,4,5,3
7,1,0,1,2,3
8,0,0,1,0,2


In [9]:
test_df.sum()

heavy            10
medium            2
light             7
sour             11
sweet            25
final_cause    5307
dtype: int64

In [10]:
# models don't seem to be working great so trying to add a general crude oil column
#test_df['general'] = ((test_df[['heavy', 'medium', 'light', 'sour', 'sweet']] == 0).all(axis=1)).astype(int)
# test_df.groupby(['final_cause']).sum()

## UPDATE: general crude oil skews the data too much; let's try running models without these records
## only 43 records... damn.
test_df = test_df.loc[test_df[['heavy', 'medium', 'light', 'sour', 'sweet']].sum(axis=1) != 0]
print(len(test_df))

43


In [11]:
## GENERATE a profiling report for the dataset that we're going to be running models on

# Define the path to the report file
report_path = Path("../reports/dataset_E_cleaned.html")

# only generate report if it doesn't already exist
if not report_path.exists():
    us_profile = ProfileReport(test_df, title="US 02-09 cleaned", explorative=True)
    us_profile.to_file(report_path)
else:
    print("Report already exists.")

## RESULTS
# light and sour highly correlated
# duplicate rows which need to be careful of based on model

Report already exists.


# ML Models

## Prepare data

In [12]:
test_df.groupby(['final_cause']).count()

,heavy,medium,light,sour,sweet
final_cause,,,,,
1,17,17,17,17,17
2,2,2,2,2,2
3,1,1,1,1,1
5,6,6,6,6,6
6,10,10,10,10,10
7,5,5,5,5,5
8,2,2,2,2,2


In [13]:
# split into attributes and target
X = test_df[['heavy', 'medium', 'light', 'sour', 'sweet']]
y = test_df['final_cause']

# split into train and test
## usually would only train on 20-30% but cant train on 10 records...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42)

## Decision Tree

In [14]:
# initialize decision tree
dt_clf = DecisionTreeClassifier(random_state=42)

# train & make predictions
dt_clf.fit(X_train, y_train)
y_pred_dt = dt_clf.predict(X_test)

# Model Eval
print("Decision Tree Classifier Report:")
print(classification_report(y_test, y_pred_dt))
print("Accuracy:", accuracy_score(y_test, y_pred_dt))

Decision Tree Classifier Report:
              precision    recall  f1-score   support

           1       0.59      0.67      0.62        15
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           5       0.38      0.75      0.50         4
           6       0.40      0.29      0.33         7
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00         2

    accuracy                           0.43        35
   macro avg       0.19      0.24      0.21        35
weighted avg       0.37      0.43      0.39        35

Accuracy: 0.42857142857142855


/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is il

## Random Forest

In [15]:
# initialize RF
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

# train & make predictions
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

# Model Evaluation
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Classifier Report:
              precision    recall  f1-score   support

           1       0.59      0.67      0.62        15
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           5       0.38      0.75      0.50         4
           6       0.40      0.29      0.33         7
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00         2

    accuracy                           0.43        35
   macro avg       0.19      0.24      0.21        35
weighted avg       0.37      0.43      0.39        35

Accuracy: 0.42857142857142855


/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is il

## Multinomial Logistic Regression

In [16]:
# initialize multinomial LR
log_reg = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)

# train & predict
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

# Model Evaluation
print("Multinomial Logistic Regression Report:")
print(classification_report(y_test, y_pred_log_reg))
print("Accuracy:", accuracy_score(y_test, y_pred_log_reg))

Multinomial Logistic Regression Report:
              precision    recall  f1-score   support

           1       0.59      0.67      0.62        15
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           5       0.38      0.75      0.50         4
           6       0.30      0.43      0.35         7
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00         2

    accuracy                           0.46        35
   macro avg       0.18      0.26      0.21        35
weighted avg       0.35      0.46      0.40        35

Accuracy: 0.45714285714285713


/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is il

# Conclusion

In [17]:
## CONCLUSION
# not enough data for ML models :(
## general observations: sweet occurs the most often - could lead further investigation towards sulfur causing corrosion